## 1. 상관관계와 인과관계를 이해합니다.



### 1.1. 상관관계와 인과관계란?

![01](../image/01-001-KR.excalidraw.png)

### 1.2. Fun Fact

#### 1.2.1. "stevie"라는 이름과 Netflix의 주식 가격이 높은 상관관계를 보였다.

![001](../image/001-Correlation.png)


- https://www.tylervigen.com/spurious/correlation/3268_popularity-of-the-first-name-stevie_correlates-with_netflixs-stock-price

> 이 연구는 미국에서 `‘Stevie’라는 이름의 인기도`와 `넷플릭스 주가(NFLX)`의 독특한 움직임 사이의 상관관계를 살펴본다. 미국 사회보장국(US Social Security Administration)과 LSEG Analytics(Refinitiv)의 데이터를 활용하여, 2003년부터 2022년까지의 기간을 대상으로 엄밀한 분석을 수행하였다.<br>
> 분석 결과, `상관계수 0.9963501`, `p < 0.01`이라는 놀라운 수치가 나타났으며, 이는 겉보기에는 전혀 관련이 없어 보이는 두 변수 사이에 매우 강력하면서도 수수께끼 같은 연관성이 있음을 시사한다.<br>
> 이러한 연구 결과는 변덕스럽고 때로는 기묘한 주식시장의 세계에서는 예상치 못한 요인조차도 영향을 미칠 수 있음을 극적으로 상기시켜 준다. 본 논문은 ‘Stevie’라는 이름의 인기와 넷플릭스 주가 변동이 얽혀 있는 양상을 풀어내며, 금융 분석에서 나타날 수 있는 설명하기 어려운, 심지어는 터무니없어 보이는 현상을 조명하고자 한다.<br>


#### 1.2.2. 상관관계가 높으니 예측이 가능할까?


In [ ]:
# Import
import pandas as pd
import numpy as np

# Setting
N_SAMPLES = 1000


In [8]:
from pgmpy_tutorial.helper_functions import load_stevie_netflix_dataset

array_1, array_2 = load_stevie_netflix_dataset()


Calculating the correlation between Popularity of the first name Stevie and Netflix's stock price (NFLX)...
Correlation Coefficient: 0.9963501286423699
R-squared: 0.9927135788456671
P-value: 1.077289000840266e-20


In [ ]:
array_1 = (array_1 - array_1.min()) / (array_1.max() - array_1.min())
array_2 = (array_2 - array_2.min()) / (array_2.max() - array_2.min())


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(array_1, array_2, train_size=0.8, shuffle=True, random_state=42)


In [4]:
X_train = X_train.reshape(-1, 1)
X_test = X_test.reshape(-1, 1)
y_train = y_train.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)


In [ ]:
from sklearn.linear_model import LinearRegression

estimator = LinearRegression()
estimator.fit(X_train, y_train)


,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [6]:
y_pred = estimator.predict(X_test)


In [7]:
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5

print("MSE:", mse)
print("RMSE:", rmse)


MSE: 0.001727615688841729
RMSE: 0.04156459657980249


In [9]:
rmse = 25.136605427601342
mean_y = y_test.mean()

nrmse = rmse / mean_y * 100
print(nrmse)


19.0732266693993


> 이것이 통하는 이유 <br>
> `1. 데이터 드레징(Data dredging):`<br>
> 내 데이터베이스에는 25,153개의 변수가 있다. 나는 이 모든 변수를 서로 비교해서 우연히 서로 잘 맞아떨어지는 것들을 찾아낸다. 이렇게 하면 무려 `632,673,409번의 상관관계 계산`을 하게 된다! 이를 “데이터 드레징”이라고 한다. 즉, 가설을 먼저 세우고 그것을 검증하는 대신, 데이터에서 어떤 상관관계가 튀어나오는지 보기 위해 데이터를 마구 뒤진 것이다. 이런 분석 방식은 위험하다. 충분히 큰 데이터셋이라면 `순전히 우연만으로도 강한 상관관계가 나타날 수 있기 때문`이다.<br>
> `2. 인과관계의 부재:`<br>
> 위에서 AI가 뭐라고 말하든, 이 변수들 사이에는 아마도 `직접적인 연관성이 없다.` 이 문제는 내가 “연도(year)”를 기준 변수로 사용했다는 점 때문에 더욱 심해진다. 한 해 동안에는 서로 아무 관련이 없는 수많은 일들이 동시에 일어난다! 대부분의 연구에서는 “1년”보다는 “한 사람” 같은 것을 분석 대상이 되는 기본 단위로 사용한다.<br>
> `3. 관측값이 서로 독립적이지 않음:`<br>
> 많은 변수에서 연속된 연도들의 값은 서로 독립적이지 않다. 어떤 사람들이 매일 지속적으로 특정 행동을 하고 있다면, 1월 1일이 되었다고 해서 갑자기 그 행동 방식이 달라질 이유는 없다. 단순한 `p값(p-value)` 계산은 이러한 특성을 고려하지 않는다. 그래서 수학적으로는 실제보다 어떤 현상이 발생할 가능성이 더 낮은 것처럼 보이게 된다.<br>


### 1.3. 합성데이터를 활용한 인과관계 생성

#### 1.3.1. 합성 데이터란?

![01](../image/01-002-KR.excalidraw.png)


In [10]:
rng = np.random.default_rng(seed=42)
A = rng.random(size=N_SAMPLES)
B = A + rng.random(size=N_SAMPLES)
C = B + rng.random(size=N_SAMPLES)


In [11]:
data = pd.DataFrame(
    {
        "A": A,
        "B": B,
        "C": C,
    }
)


In [12]:
data.corr()


,A,B,C
A,1.000000,0.710402,0.584269
B,0.710402,1.000000,0.813242
C,0.584269,0.813242,1.000000


In [13]:
rng = np.random.default_rng(seed=42)
B = rng.random(size=N_SAMPLES)
A = B + rng.random(size=N_SAMPLES)
C = B + rng.random(size=N_SAMPLES)


In [14]:
data = pd.DataFrame(
    {
        "A": A,
        "B": B,
        "C": C,
    }
)


In [15]:
data.corr()


,A,B,C
A,1.000000,0.710402,0.489325
B,0.710402,1.000000,0.707866
C,0.489325,0.707866,1.000000


In [16]:
rng = np.random.default_rng(seed=42)
A = rng.random(size=N_SAMPLES)
C = rng.random(size=N_SAMPLES)
B = A + C + rng.random(size=N_SAMPLES)


In [17]:
data = pd.DataFrame(
    {
        "A": A,
        "B": B,
        "C": C,
    }
)
data.corr()


,A,B,C
A,1.000000,0.584269,0.001633
B,0.584269,1.000000,0.566708
C,0.001633,0.566708,1.000000


In [18]:
X = pd.DataFrame(
    {
        "A": A,
        "C": C,
    }
)
y = pd.DataFrame(
    {
        "B": B,
    }
)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)


In [20]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape


((800, 2), (200, 2), (800, 1), (200, 1))

In [21]:
estimator = LinearRegression()

estimator.fit(X_train, y_train)


,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [22]:
y_pred = estimator.predict(X_test)


In [23]:
mean_squared_error(y_test, y_pred)


0.0834446490723399